In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from qiskit.circuit.library import QAOAAnsatz
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from qiskit_optimization import QuadraticProgram
from scipy.optimize import minimize
%matplotlib inline

In [ ]:
# inputs
rng = np.random.default_rng(2026)
n, K, q = 6, 3, 0.5               # stocks, pick exactly K, risk aversion
mu = rng.uniform(0.05, 0.25, n)   # expected returns
A = rng.normal(0, 0.1, (n, n))
sigma = A @ A.T                   # covariance (risk)
lam = 0.5 # from earlier example

In [ ]:
# create the cost function per state
def getCost(x):
    return q * x @ sigma @ x - mu @ x + lam * (x.sum() - K) ** 2

In [ ]:
# brute force: the 20 valid portfolios and the best one
portfolios = []
for stocks in combinations(range(n), K):
    x = np.zeros(n, dtype=int)
    x[list(stocks)] = 1
    portfolios.append(x)

best_x = min(portfolios, key=getCost)
print("best portfolio:", best_x, "cost:", round(getCost(best_x), 4))

The quantum based circuit and optimisation has four key stages:

1. Creating the necessary inputs - the covariance matrices, returns per stock, etc. All the above information is provided in cell 2.

2. Create Pauli matrices. These transform the above math into quantum problems. Qiskit's SparsePauliOp is used to store these efficiently.

3. Create the circuit and run the hybrid optimisation loop. this is done in phases:

a. Create a new instance of QAOAAnsatz class (Ansatz object) and pass the cost matrices (Hamiltonian) computed in step 1.

b. Define the execution env - local or IBM Hardware (configure with API keys).

c. Run a hybrid optimisation loop - using scikit and qiskit. This is a tyuning phase where the optimal values for a specific class are defined based on iterative tuning by scikit's minimizer (like COBYLA) and qiskit estimator (computes the Hamiltonian).

d. The above generated Hamiltonian is then passed to scikit's minimizer and the key params are tuned to minimize this.

4. Once the best params are inferred, the values are substituted in the circuit designed from Step 3. The run is performed over multiple iterations (bootstraps or 'shots'), and most frequent bitstrings are recorded.

5. Using a combination of bitstrings and the original formulation (step 1), compute the results and grade their performance.

For all binary problems, the quantum mapping starts with the same formulation : **Quadratic Unconstrained Binary Optimization (QUBO) + Ising Model (The Quantum Mapping)**. Jordan-Wigner transformation or the Bravyi-Kitaev transformation are used for continuous variable transformations.

In [ ]:
linear_dict = {}
quadratic_dict = {}

# 1. Build the Linear Dictionary
for i in range(n):
    var_name = f"x_{i}"
    
    # Linear = (q * Individual Risk) - Return + Penalty_linear_term
    linear_value = (q * sigma[i][i]) - mu[i] + lam * (1 - 2 * K)
    linear_dict[var_name] = float(linear_value)

# 2. Build the Quadratic Dictionary
for i in range(n):
    for j in range(i + 1, n): # i+1 to n avoids self-loops and duplicate pairs
        pair = (f"x_{i}", f"x_{j}")
        
        # Quadratic = (2 * q * Covariance) + Penalty_quadratic_term
        # Multiply risk by 2 because the matrix is symmetric and we only loop the upper triangle
        quad_value = (2 * q * sigma[i][j]) + (2 * lam)
        quadratic_dict[pair] = float(quad_value)

# 3. Calculate Constant
constant_val = float(lam * (K ** 2))

# 4. Initialize the Model and apply the dictionaries
mod = QuadraticProgram("Portfolio_Optimization")

# Add the binary variables to the model
for i in range(n):
    mod.binary_var(name=f"x_{i}")

# Add the objective function using the dictionaries
mod.minimize(
    constant=constant_val,
    linear=linear_dict,
    quadratic=quadratic_dict
)

print(mod.prettyprint())

In [ ]:
pauliop, offset = mod.to_ising()  # Pauli operator + constant offset
print(pauliop)
print("offset:", offset)

### Creating the circuit and using IBM hardware

In [ ]:
with open('/mnt/g/quantum_qiskit_hackathon/ibm_key.txt', 'r') as f:
  api_key = f.readline()

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(
    channel="ibm_quantum_platform", # optional
    token=api_key,
    instance="crn:v1:bluemix:public:quantum-computing:us-east:a/65f9f33d235f4862a3082307230b71c8:790aac10-9e06-4821-9125-c79043f61fe0::" # recommended
    )

In [ ]:
# from qiskit.transpiler import generate_preset_pass_manager

# backend = service.least_busy(
#     operational=True, simulator=True, min_num_qubits=127
# )
# print(backend)

# # Create pass manager for transpilation. Level 3 is the most aggressive
# # preset: slower to transpile, but produces shorter circuits that are
# # more robust to hardware noise.
# pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

# candidate_circuit = pm.run(circuit)
# candidate_circuit.draw("mpl", fold=False, idle_wires=False)

In [ ]:
# QiskitRuntimeService.save_account(
#   # For `token`, use the 44-character API_KEY you created 
#   # and saved from the IBM Quantum Platform Home dashboard:
#   token=api_key, 
#   name="SreeramPeela", # Optional
#   instance="crn:v1:bluemix:public:quantum-computing:us-east:a/65f9f33d235f4862a3082307230b71c8:790aac10-9e06-4821-9125-c79043f61fe0::", # Optional
#   set_as_default=True, # Optional
#   overwrite=False, # Optional
# )

In [ ]:
# Map the Cost Hamiltonian to the physical qubits chosen during transpilation
# isa_observable = pauliop.apply_layout(candidate_circuit.layout)

In [ ]:
# # 1. Initialize tracking list
# objective_func_vals = []

# def cost_function(angles):
#     job = estimator.run([(candidate_circuit, isa_observable, angles)])
#     result = job.result()[0]
    
#     # Extract expectation value and cast to float
#     energy = float(result.data.evs)
    
#     # Append each evaluated cost
#     objective_func_vals.append(energy)
#     return energy

In [ ]:
# 1. Set up the circuit and estimator
estimator_circuit = QAOAAnsatz(cost_operator=pauliop, reps=2)
local_estimator = StatevectorEstimator()

objective_func_vals = []  # energy history for the convergence plot

# 2. Cost function: expected energy <H> for the given angles
def cost_function(angles):
    job = local_estimator.run([(estimator_circuit, pauliop, angles)])
    energy = float(job.result()[0].data.evs)
    objective_func_vals.append(energy)
    return energy

# 3. Run the optimization loop locally
num_params = estimator_circuit.num_parameters
initial_angles = np.random.rand(num_params) * np.pi

print("Starting optimization...")
optimization_result = minimize(cost_function, initial_angles, method="COBYLA")
print(f"Optimization finished! Best angles: {optimization_result.x}")

# 4. Plot the convergence curve
plt.figure(figsize=(10, 5))
plt.plot(objective_func_vals, marker="o", markersize=3, linestyle="-", color="tab:blue")
plt.title("QAOA Convergence (Energy vs. Function Evaluations)")
plt.xlabel("Function Evaluation Index")
plt.ylabel("Cost / Energy $\\langle H \\rangle$")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

In [ ]:
# 1. Create a measured copy of the circuit for the Sampler
sampler_circuit = estimator_circuit.copy()
sampler_circuit.measure_all()

# 2. Bind the optimal angles found by SciPy
bound_circuit = sampler_circuit.assign_parameters(optimization_result.x)

# 3. Sample the final portfolios locally
local_sampler = StatevectorSampler()
job = local_sampler.run([bound_circuit], shots=4096)
counts = job.result()[0].data.meas.get_counts()
print(counts)

In [ ]:
# keep only portfolios that buy exactly K stocks, most frequent first
valid_counts = {k: v for k, v in counts.items() if k.count('1') == K}
sorted_valid_counts = dict(sorted(valid_counts.items(), key=lambda item: item[1], reverse=True))

print(sorted_valid_counts)

In [ ]:
# Qiskit keys are little-endian (qubit 0 is the rightmost character): reverse to get x_0..x_5
top = list(sorted_valid_counts.keys())[0][::-1]
print(top, "matches brute force:", top == "".join(map(str, best_x)))

The above one is same as the one determined by the brute force algorithm earlier.